# E-Commerce 360 — Stage 1: Data Understanding

**Dataset:** Olist Brazilian E-Commerce Public Dataset (9 CSV files, Brazilian public data)
**Goal of this notebook:** understand every table before touching it: what one row means (grain), which columns identify a row (keys), how tables connect (relationships), and what is wrong with the data (quality issues).

**Rule:** raw data is only *read* here. Nothing is filled, dropped, renamed or type-converted. Every issue is recorded and handled in Stage 2 (Data Preparation).

**Fixed workflow used for every table:**
1. Load and preview
2. Basic checks: shape, info, missing values, complete duplicate rows
3. Key / uniqueness and grain
4. Relationship checks with connected tables
5. Investigate only the anomalies found above
6. Observations (short, fixed format)

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")          # raw files stay untouched

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Customers

### 1.1 Load and preview

In [2]:
customers = pd.read_csv("data_raw/olist_customers_dataset.csv")
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


### 1.2 Basic checks

In [3]:
customers.shape

(99441, 5)

In [4]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [5]:
customers.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [6]:
customers.duplicated().sum()

np.int64(0)

### 1.3 Key / uniqueness and grain

In [7]:
customers[["customer_id", "customer_unique_id"]].nunique()

customer_id           99441
customer_unique_id    96096
dtype: int64

In [8]:
(customers.groupby("customer_unique_id")["customer_id"].nunique() > 1).sum()

np.int64(2997)

### Customers: Observations

- 99,441 rows and 5 columns; no missing values and no complete duplicate rows.
- `customer_id` is unique for every row (99,441), so it is the table key.
- `customer_unique_id` has only 96,096 distinct values: 2,997 real customers appear under more than one `customer_id` (3,345 extra ids in total).
- Meaning: `customer_id` is created per order, `customer_unique_id` identifies the actual person. **Repeat-customer analysis must use `customer_unique_id`.**
- The link between customers and orders is checked in the Orders section.

## 2. Orders

### 2.1 Load and preview

In [9]:
orders = pd.read_csv("data_raw/olist_orders_dataset.csv")
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### 2.2 Basic checks

In [10]:
orders.shape

(99441, 8)

In [11]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [12]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [13]:
orders.duplicated().sum()

np.int64(0)

### 2.3 Key / uniqueness, status and relationship

In [14]:
orders[["order_id", "customer_id"]].nunique()

order_id       99441
customer_id    99441
dtype: int64

In [15]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [16]:
orders["customer_id"].isin(customers["customer_id"]).value_counts()

customer_id
True    99441
Name: count, dtype: int64

### 2.4 Date range and missing timestamps

In [17]:
orders[["order_purchase_timestamp", "order_estimated_delivery_date"]].agg(["min", "max"])

,order_purchase_timestamp,order_estimated_delivery_date
min,2016-09-04 21:15:19,2016-09-30 00:00:00
max,2018-10-17 17:30:18,2018-11-12 00:00:00


In [18]:
date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
]

missing_dates_by_status = pd.DataFrame({
    column: orders.loc[orders[column].isna(), "order_status"].value_counts()
    for column in date_columns
}).fillna(0).astype(int)

missing_dates_by_status

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [19]:
delivered = orders["order_status"] == "delivered"
orders.loc[delivered, date_columns].isna().sum()

order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
dtype: int64

### 2.5 Timestamp logic checks (temporary datetime variables; the table itself is not changed)

In [20]:
purchase_ts = pd.to_datetime(orders["order_purchase_timestamp"])
carrier_ts = pd.to_datetime(orders["order_delivered_carrier_date"])
customer_ts = pd.to_datetime(orders["order_delivered_customer_date"])

In [21]:
(carrier_ts < purchase_ts).sum()

np.int64(166)

In [22]:
(customer_ts < purchase_ts).sum()

np.int64(0)

In [23]:
(customer_ts < carrier_ts).sum()

np.int64(23)

### Orders: Observations

- 99,441 rows and 8 columns; no complete duplicate rows.
- `order_id` is unique, so the grain is **one row per order**. `customer_id` is also unique here, which confirms it is generated per order.
- All order `customer_id` values exist in Customers (no orphans).
- Purchase timestamps run from 2016-09-04 to 2018-10-17; estimated delivery dates run to 2018-11-12.
- Orders are not all delivered; the status breakdown must be considered before calculating sales and delivery KPIs.
- Missing timestamps: approval 160, carrier handover 1,783, customer delivery 2,965. Among `delivered` orders, 14, 2 and 8 rows respectively miss these dates.
- Timestamp logic: 166 orders show carrier handover **before** purchase (165 of them `delivered`), and 23 show customer delivery before carrier handover. No delivery is dated before purchase.
- Nothing is converted or filled here; these rows are flagged for Stage 2.

## 3. Order Items

### 3.1 Load and preview

In [24]:
order_items = pd.read_csv("data_raw/olist_order_items_dataset.csv")
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


### 3.2 Basic checks

In [25]:
order_items.shape

(112650, 7)

In [26]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [27]:
order_items.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [28]:
order_items.duplicated().sum()

np.int64(0)

### 3.3 Key / uniqueness and grain

In [29]:
order_items[["order_id", "order_item_id"]].duplicated().sum()

np.int64(0)

In [30]:
order_items["order_id"].nunique()

98666

In [31]:
order_items["order_id"].value_counts().max()

np.int64(21)

### 3.4 Relationship and value-range checks

In [32]:
order_items["order_id"].isin(orders["order_id"]).value_counts()

order_id
True    112650
Name: count, dtype: int64

In [33]:
orders.loc[~orders["order_id"].isin(order_items["order_id"]), "order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [34]:
order_items[["price", "freight_value"]].agg(["min", "max"])

,price,freight_value
min,0.85,0.00
max,6735.00,409.68


In [35]:
(order_items["price"] <= 0).sum()

np.int64(0)

### Order Items: Observations

- 112,650 rows and 7 columns; no missing values and no complete duplicate rows.
- `order_id` + `order_item_id` has no duplicates, so the grain is **one row per item within an order**. This is the table that carries revenue (`price`, `freight_value`).
- 98,666 distinct orders; one order has up to 21 items.
- All `order_id` values exist in Orders.
- 775 orders have no item rows; none of them is `delivered` (603 unavailable, 164 canceled, 5 created, 2 invoiced, 1 shipped).
- `price` ranges from 0.85 to 6,735.00 (no zero or negative prices); `freight_value` ranges from 0 to 409.68.
- Because one order has many item rows, joining Orders to Order Items multiplies order rows. Order counts must use distinct `order_id`.

## 4. Products

### 4.1 Load and preview

In [36]:
products = pd.read_csv("data_raw/olist_products_dataset.csv")
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


### 4.2 Basic checks

In [37]:
products.shape

(32951, 9)

In [38]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [39]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [40]:
products.duplicated().sum()

np.int64(0)

### 4.3 Key / uniqueness and relationship

In [41]:
products["product_id"].is_unique

True

In [42]:
order_items["product_id"].isin(products["product_id"]).value_counts()

product_id
True    112650
Name: count, dtype: int64

In [43]:
products["product_id"].isin(order_items["product_id"]).value_counts()

product_id
True    32951
Name: count, dtype: int64

### 4.4 Missing-data investigation

In [44]:
products.isna().any(axis=1).sum()

np.int64(611)

In [45]:
incomplete_product_ids = products.loc[products.isna().any(axis=1), "product_id"]
order_items["product_id"].isin(incomplete_product_ids).sum()

np.int64(1604)

In [46]:
products[["product_category_name", "product_weight_g"]].isna().sum()

product_category_name    610
product_weight_g           2
dtype: int64

### Products: Observations

- 32,951 rows and 9 columns; no complete duplicate rows.
- `product_id` is unique, so the grain is **one row per product**.
- All products in Order Items exist in Products, and every product is sold at least once.
- 611 products have at least one missing attribute: 610 miss category, name length, description length and photo count; 2 miss weight and dimensions (one product has both problems).
- These incomplete products appear in 1,604 order-item rows, so they cannot be ignored safely.
- Column names `product_name_lenght` and `product_description_lenght` are misspelled in the source file; they are kept as-is here and can be renamed in Stage 2.

## 5. Sellers

### 5.1 Load and preview

In [47]:
sellers = pd.read_csv("data_raw/olist_sellers_dataset.csv")
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


### 5.2 Basic checks

In [48]:
sellers.shape

(3095, 4)

In [49]:
sellers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


In [50]:
sellers.isnull().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [51]:
sellers.duplicated().sum()

np.int64(0)

### 5.3 Key / uniqueness and relationship

In [52]:
sellers["seller_id"].is_unique

True

In [53]:
order_items["seller_id"].isin(sellers["seller_id"]).value_counts()

seller_id
True    112650
Name: count, dtype: int64

In [54]:
sellers["seller_id"].isin(order_items["seller_id"]).value_counts()

seller_id
True    3095
Name: count, dtype: int64

### Sellers: Observations

- 3,095 rows and 4 columns; no missing values and no complete duplicate rows.
- `seller_id` is unique, so the grain is **one row per seller**.
- All sellers in Order Items exist in Sellers, and every seller has at least one item sold.
- Seller performance and concentration are analysed later in Business Analytics, not here.

## 6. Order Payments

### 6.1 Load and preview

In [55]:
payments = pd.read_csv("data_raw/olist_order_payments_dataset.csv")
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


### 6.2 Basic checks

In [56]:
payments.shape

(103886, 5)

In [57]:
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [58]:
payments.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [59]:
payments.duplicated().sum()

np.int64(0)

### 6.3 Key / uniqueness and relationship

In [60]:
payments["order_id"].nunique()

99440

In [61]:
payments[["order_id", "payment_sequential"]].duplicated().sum()

np.int64(0)

In [62]:
payments["order_id"].value_counts().max()

np.int64(29)

In [63]:
payments["order_id"].isin(orders["order_id"]).value_counts()

order_id
True    103886
Name: count, dtype: int64

In [64]:
orders_without_payment = orders.loc[
    ~orders["order_id"].isin(payments["order_id"]),
    ["order_id", "order_status"],
].copy()

orders_without_payment["order_item_rows"] = (
    orders_without_payment["order_id"].map(order_items.groupby("order_id").size())
)
orders_without_payment

,order_id,order_status,order_item_rows
30710,bfbd0f9bdef84302105ad712db648a6c,delivered,3


### 6.4 Payment-specific checks

In [65]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [66]:
payments[["payment_value", "payment_installments"]].agg(["min", "max"])

,payment_value,payment_installments
min,0.00,0
max,13664.08,24


In [67]:
payments_to_review = payments.loc[
    (payments["payment_value"] == 0)
    | (payments["payment_installments"] == 0)
    | (payments["payment_type"] == "not_defined")
].merge(orders[["order_id", "order_status"]], on="order_id", how="left")

payments_to_review.sort_values(["order_status", "payment_type"])

,order_id,payment_sequential,payment_type,payment_installments,payment_value,order_status
4,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00,canceled
5,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00,canceled
9,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00,canceled
3,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69,delivered
8,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94,delivered
0,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00,delivered
2,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00,delivered
6,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00,delivered
10,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.00,delivered
1,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00,shipped


### Payments: Observations

- 103,886 rows and 5 columns; no missing values and no complete duplicate rows.
- `order_id` + `payment_sequential` has no duplicates, so the grain is **one row per payment record**. An order can have several (up to 29).
- Only 99,440 distinct orders have payments, against 103,886 payment rows. **Joining payments directly to order items would multiply revenue**; payments must be aggregated to order level first.
- All payment `order_id` values exist in Orders. One `delivered` order has no payment record (it has 3 item rows).
- Payment types: `credit_card`, `boleto`, `voucher`, `debit_card`, plus 3 `not_defined` rows (all on canceled orders, value 0).
- 9 rows have payment value 0 (6 vouchers and the 3 `not_defined`). 2 delivered credit-card rows have 0 installments despite a non-zero value.

## 7. Order Reviews

### 7.1 Load and preview

In [68]:
reviews = pd.read_csv("data_raw/olist_order_reviews_dataset.csv")
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


### 7.2 Basic checks

In [69]:
reviews.shape

(99224, 7)

In [70]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [71]:
reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [72]:
reviews.duplicated().sum()

np.int64(0)

### 7.3 Key / uniqueness and relationship

In [73]:
reviews["review_id"].nunique()

98410

In [74]:
reviews["review_id"].duplicated().sum()

np.int64(814)

In [75]:
reviews["order_id"].nunique()

98673

In [76]:
reviews.duplicated(subset=["review_id", "order_id"]).sum()

np.int64(0)

In [77]:
reviews["order_id"].isin(orders["order_id"]).value_counts()

order_id
True    99224
Name: count, dtype: int64

In [78]:
(~orders["order_id"].isin(reviews["order_id"])).sum()

np.int64(768)

### 7.4 Review-specific checks

In [79]:
reviews["order_id"].value_counts().value_counts().sort_index()

count
1    98126
2      543
3        4
Name: count, dtype: int64

In [80]:
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

### Reviews: Observations

- 99,224 rows and 7 columns; no complete duplicate rows.
- `review_id` is **not** unique: 98,410 distinct values, 764 ids occur twice and 25 occur three times, attached to different orders. `review_id` + `order_id` has no duplicates and is used as the key.
- `order_id` is also not unique: 98,673 distinct orders. 547 orders have more than one review (543 with two, 4 with three).
- All review `order_id` values exist in Orders. 768 orders have no review at all.
- `review_score` ranges from 1 to 5.
- Comment title is missing in 87,656 rows and comment message in 58,247 rows; comments are optional.
- **Joining reviews to orders without handling repeats would duplicate orders and distort average review score.** A rule to keep one review per order is needed in Stage 2.

## 8. Geolocation

### 8.1 Load and preview

In [81]:
geolocation = pd.read_csv("data_raw/olist_geolocation_dataset.csv")
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


### 8.2 Basic checks

In [82]:
geolocation.shape

(1000163, 5)

In [83]:
geolocation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB


In [84]:
geolocation.isnull().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [85]:
geolocation.duplicated().sum()

np.int64(261831)

### 8.3 Key / uniqueness and relationships

In [86]:
geolocation["geolocation_zip_code_prefix"].nunique()

19015

In [87]:

customers["customer_zip_code_prefix"].isin(geolocation["geolocation_zip_code_prefix"]).value_counts()

customer_zip_code_prefix
True     99163
False      278
Name: count, dtype: int64

In [88]:
sellers["seller_zip_code_prefix"].isin(geolocation["geolocation_zip_code_prefix"]).value_counts()

seller_zip_code_prefix
True     3088
False       7
Name: count, dtype: int64

### 8.4 ZIP-code mapping check

In [89]:
labels_per_zip = geolocation.groupby("geolocation_zip_code_prefix")[
    ["geolocation_city", "geolocation_state"]
].nunique()

labels_per_zip.gt(1).any(axis=1).sum()

np.int64(8559)

### Geolocation: Observations

- 1,000,163 rows and 5 columns; no missing values.
- 261,831 rows are complete duplicates (about 26%).
- Only 19,015 distinct ZIP prefixes, so the grain is **many coordinate rows per ZIP prefix** and there is no unique key.
- 278 customers and 7 sellers have a ZIP prefix that does not exist in Geolocation.
- 8,559 ZIP prefixes are linked to more than one city/state label.
- **Joining customers or sellers to this table on ZIP prefix would multiply rows.** Stage 2 must first reduce it to one row per ZIP prefix.

## 9. Product Category Translation

### 9.1 Load and preview

In [90]:
category_translation = pd.read_csv("data_raw/product_category_name_translation.csv")
category_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


### 9.2 Basic checks

In [91]:
category_translation.shape

(71, 2)

In [92]:
category_translation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


In [93]:
category_translation.isnull().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

In [94]:
category_translation.duplicated().sum()

np.int64(0)

### 9.3 Key / uniqueness and relationship

In [95]:
category_translation["product_category_name"].is_unique

True

In [96]:
product_categories = products["product_category_name"].dropna().drop_duplicates()
product_categories.isin(category_translation["product_category_name"]).value_counts()

product_category_name
True     71
False     2
Name: count, dtype: int64

In [97]:
product_categories[~product_categories.isin(category_translation["product_category_name"])]

1628                                         pc_gamer
5821    portateis_cozinha_e_preparadores_de_alimentos
Name: product_category_name, dtype: object

### Category Translation: Observations

- 71 rows and 2 columns; no missing values and no duplicates.
- `product_category_name` is unique, so the grain is **one row per Portuguese category**.
- Products use 73 distinct categories; 71 have an English translation.
- `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos` have no translation (13 product rows in total).

## 10. Stage 1 Summary

### 10.1 Data model: keys and relationships (verified against the data)

| Child table | Foreign key | Parent table | Relationship | Verified result |
|---|---|---|---|---|
| orders | customer_id | customers | 1 : 1 | no orphans |
| order_items | order_id | orders | many : 1 | no orphans; 775 orders without items |
| order_items | product_id | products | many : 1 | no orphans |
| order_items | seller_id | sellers | many : 1 | no orphans |
| payments | order_id | orders | many : 1 | no orphans; 1 order without payment |
| reviews | order_id | orders | many : 1 | no orphans; 768 orders without review |
| products | product_category_name | category_translation | many : 1 | 2 untranslated categories |
| customers / sellers | zip_code_prefix | geolocation | many : many | ZIP is not unique in geolocation |

### 10.2 Issue log for Stage 2 (priority order)

| Priority | Issue | Size | Risk if ignored | Planned Stage 2 action |
|---|---|---|---|---|
| High | Reviews repeated per order | 547 orders | duplicated orders, wrong review score | keep one review per order |
| High | Payments have several rows per order | 99,440 orders, 103,886 rows | revenue multiplied in joins | aggregate payments to order level first |
| High | `customer_id` is per order | 2,997 people with several ids | repeat customers undercounted | use `customer_unique_id` |
| High | Geolocation: duplicates and multi-label ZIPs | 261,831 duplicate rows; 8,559 ZIPs | rows multiplied in joins | one row per ZIP prefix |
| Medium | Missing / invalid delivery timestamps | 8 delivered without customer date; 166 carrier date before purchase | wrong delivery-time KPIs | exclude from delivery KPIs |
| Medium | Products without category | 610 products | category analysis gaps | label as `unknown` |
| Medium | Delivered order with no payment | 1 order | minor payment mismatch | keep, flag |
| Low | Zero payment value / `not_defined` / 0 installments | 9 / 3 / 2 rows | minor | recode or flag |
| Low | Untranslated categories | 2 categories | missing English names | translate manually |
| Low | Missing weight and dimensions | 2 products | none for core KPIs | leave null |

### 10.3 Dataset limitations identified so far

- Public historical data (2016-09 to 2018-10) from a Brazilian marketplace; it does not describe current business performance.
- Customers are tracked per order id; true repeat behaviour is only visible through `customer_unique_id`.
- Only orders with item rows can carry revenue; 775 orders have none.
- Findings from this data describe association, not causation.

**Stage 1 is complete.** Next: Stage 2, Data Preparation and Data Modelling.